In [14]:
from dotenv import load_dotenv
from datetime import datetime, timedelta

import os
import json
import openai
import requests

# Load environment variables from .env file
load_dotenv('~/Documents/Jupyter/MDST/WN25/Eventure/EventureApp/.env')

openai.api_key = os.getenv("OPENAI_API_KEY")
ticketmaster_key = os.getenv("TICKETMASTER_API_KEY")

# Verify
print("OpenAI Key Found:", bool(os.getenv("OPENAI_API_KEY")))

MODEL_NAME = "gpt-4o-mini"
client = openai.OpenAI()


OpenAI Key Found: True


Create Functions:

In [15]:
# def makeTicketmasterEventRequest(eventparams: str):
#     # Format for looking at events: https://app.ticketmaster.com/{package}/{version}/events.json?{eventparams}apikey={API key}
#     # eventparams needs to be separated by & symbols for each query. Check ticketmaster discovery page to see all query options

#     url = f'https://app.ticketmaster.com/discovery/v2/events.json?{eventparams}&apikey={ticketmaster_key}'

#     r = requests.get(url)

#     return r 

def makeTicketmasterEventRequest(params: dict):
    print(f"--- Calling Ticketmaster API with params: {params} ---")
    params['apikey'] = ticketmaster_key
    url = f'https://app.ticketmaster.com/discovery/v2/events.json'

    r = requests.get(url,params=params)
    print(f"--- Ticketmaster API Status Code: {r.status_code} ---")

    return r.text

Describe functions to OpenAI:

In [16]:
function_descriptions = [
    {
        "type": "function",
        "function": {
            "name": "makeTicketmasterEventRequest",
            "description": "Fetches events happening in a specific location from Ticketmaster.",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "Search term for events, e.g., 'music', 'sports'."
                    },
                    "city": {
                        "type": "string",
                        "description": "The city to fetch events for, e.g., 'Detroit'."
                    },
                    "stateCode": {
                        "type": "string",
                        "description": "The state code, e.g., 'MI' for Michigan."
                    },
                    "countryCode": {
                        "type": "string",
                        "description": "The country code, e.g., 'US'."
                    },
                    "startDateTime": {
                        "type": "string",
                        "format": "date-time",
                        "description": "Start date and time for events in ISO 8601 format (YYYY-MM-DDTHH:MM:SSZ)."
                    },
                    "endDateTime": {
                        "type": "string",
                        "format": "date-time",
                        "description": "End date and time for events in ISO 8601 format."
                    }
                },
                "required": ["city", "stateCode", "countryCode"]
            }
        }
    }
]


In [17]:
def test_call_model(user_message: str, function_call="auto"):
    """
    1) We send user_message + function_descriptions to the model
    2) We see if it returns a function call
    3) If so, we parse arguments, call the function ourselves
    4) Return final output
    """

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": user_message}],
        tools=function_descriptions,  # 'functions' is now called 'tools'
        tool_choice=function_call,    # 'function_call' is now 'tool_choice'
    )
    response = completion.choices[0].message
    return response

In [18]:
def run_conversation(user_prompt_1):
  messages = []
  messages.append({"role": "user", "content": user_prompt_1})

  response_message = test_call_model(user_prompt_1)
  messages.append(response_message.model_dump(exclude_unset=True))

  tool_calls = response_message.tool_calls
  
  available_functions = {
    "makeTicketmasterEventRequest": makeTicketmasterEventRequest,
    # we can add more functions here <--
  }

  tool_responses = []

  for tool_call in tool_calls:
    function_name = tool_call.function.name
    tool_call_id = tool_call.id

    function_to_call = available_functions[function_name]

    function_args = json.loads(tool_call.function.arguments)
    function_response_data = function_to_call(params=function_args)

    tool_responses.append(
        {
          "tool_call_id": tool_call_id,
          "role": "tool",
          "name": function_name,
          "content": function_response_data,
      }
    )

  messages.extend(tool_responses)

  second_response = client.chat.completions.create(
      model=MODEL_NAME,
      messages=messages,
  )
  final_response_message = second_response.choices[0].message
  messages.append(final_response_message.model_dump(exclude_unset=True))
  final_response_content = final_response_message.content

  return final_response_content, messages

In [ ]:
from datetime import date, timedelta
today = date.today()
start_date = today.strftime("%Y-%m-%dT00:00:00Z")
end_date = (today + timedelta(days=7)).strftime("%Y-%m-%dT23:59:59Z")

answer, current_history = run_conversation(
    f"What events are going on in Michigan between {start_date} and {end_date}?",
)
print(f"Answer: {answer}")

--- Calling Ticketmaster API with params: {'city': 'Detroit', 'stateCode': 'MI', 'countryCode': 'US', 'startDateTime': '2025-03-28T00:00:00Z', 'endDateTime': '2025-04-04T23:59:59Z'} ---
--- Ticketmaster API Status Code: 200 ---
Answer: Here's a list of events happening in Michigan, particularly in Detroit, between March 28, 2025, and April 4, 2025:

### Events

1. **Detroit Pistons vs. Cleveland Cavaliers (Pride Giveaway)**
   - **Date**: March 28, 2025
   - **Time**: 7:00 PM
   - **Venue**: Little Caesars Arena
   - **Tickets**: [View Event](https://www.ticketmaster.com/detroit-pistons-v-cleveland-cavaliers-pride-detroit-michigan-03-28-2025/event/0800610E0DA84E21)
   ![](https://s1.ticketm.net/dam/a/9e8/26685b57-d741-40e1-b93d-e780b38a79e8_ARTIST_PAGE_3_2.jpg)

2. **Detroit Red Wings vs. Boston Bruins**
   - **Date**: March 29, 2025
   - **Time**: 8:00 PM
   - **Venue**: Little Caesars Arena
   - **Tickets**: [View Event](https://www.ticketmaster.com/detroit-red-wings-vs-boston-bruins

In [20]:
# Let's do a quick test with something that calls 'get_weather_info'
# user_prompt_1 = "What events are going on in Michigan for this month"
# resp = test_call_model(user_prompt_1, function_call="auto")
# print("Model response:\n", resp)


## **TODO:** Fix the OpenAI Instance 
- Fix the 'function_call=None' to be 'function_call=makeTicketmasterEventRequest'
- Assumed because 'function_descriptions' is wrong, or 'makeTicketmasterEventRequest' is broken